# RAG With LangChain

**Author:** Shinin Varongchayakul

**Date:** 21 Jun 2026

## 1. Load Documents

In [1]:
# Import packages
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

# Initialise loader
loader = DirectoryLoader(
    path="documents",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

# Load documents
docs = loader.load()

/var/folders/rd/488wg5_j58vd_vd5mpxn1xx40000gn/T/ipykernel_12737/163320869.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


In [2]:
# View loaded documents
for doc in docs:
    print("=" * 50)
    print(doc.metadata["source"])
    print("=" * 50)
    print(doc.page_content[:200])

documents/remote_work_policy.txt
DataWise Co. Remote Work Policy

Employees may work from home up to 2 days per week.

Remote work must be approved by the employee's direct manager.

Employees must be reachable on Slack during core w
documents/benefits_policy.txt
DataWise Co. Benefits Policy

Full-time employees receive health insurance after completing probation.

The company provides annual health checkups once per year.

Employees can claim up to 2,000 THB 
documents/compensation_policy.txt
DataWise Co. Compensation Policy

Salary is paid on the last working day of each month.

Performance bonuses are reviewed once per year in December.

Employees may receive an annual salary adjustment 
documents/leave_policy.txt
DataWise Co. Leave Policy

Full-time employees receive 10 days of annual leave per year after completing probation.

Employees receive 15 days of paid sick leave per year.

Sick leave of 3 consecutive


## 2. Embed Documents

### 2.1 Split Text

In [3]:
# Import package
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Split documents
chunks = text_splitter.split_documents(docs)

In [4]:
# View results
for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i+1}")
    print("Source:", chunk.metadata["source"])
    print(chunk.page_content)
    print("-" * 50)

Chunk 1
Source: documents/remote_work_policy.txt
DataWise Co. Remote Work Policy

Employees may work from home up to 2 days per week.

Remote work must be approved by the employee's direct manager.

Employees must be reachable on Slack during core working hours from 10:00 AM to 4:00 PM.

Employees working remotely are responsible for maintaining a stable internet connection and a quiet work environment.

New employees may request remote work only after completing their first month.
--------------------------------------------------
Chunk 2
Source: documents/benefits_policy.txt
DataWise Co. Benefits Policy

Full-time employees receive health insurance after completing probation.

The company provides annual health checkups once per year.

Employees can claim up to 2,000 THB per month for wellness activities such as fitness memberships, yoga classes, or mental health support.

Employees are also eligible for learning support. The company reimburses up to 10,000 THB per year for approved 

### 2.2 Create Embedder

In [ ]:
# Get Gemini API key

# Import packages
import os
from pathlib import Path
from dotenv import load_dotenv

# Specify root folder
ROOT_DIR = Path.cwd().parents[2]

# Get .env path
ENV_PATH = ROOT_DIR / ".env"

# Load .env
load_dotenv(ROOT_DIR / ".env")

# Get API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [6]:
# Import package
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Create embedder
document_embedder = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    task_type="retrieval_document",
    google_api_key=GEMINI_API_KEY
)

### 2.3 Store Embeddings

In [7]:
# Import package
from langchain_community.vectorstores import FAISS

# Build vector DB
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=document_embedder
)

## 3. Create Retriever

In [8]:
# Creater retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [9]:
# Test retriever
question = "Do I need a medical certificate for sick leave?"

relevant_docs = retriever.invoke(question)

for i, doc in enumerate(relevant_docs, start=1):
    print(f"Retrieved chunk {i}")
    print("Source:", doc.metadata["source"])
    print(doc.page_content)
    print("-" * 60)

Retrieved chunk 1
Source: documents/leave_policy.txt
DataWise Co. Leave Policy

Full-time employees receive 10 days of annual leave per year after completing probation.

Employees receive 15 days of paid sick leave per year.

Sick leave of 3 consecutive days or more requires a medical certificate.

Employees should submit annual leave requests at least 7 days in advance through the HR system.

Unused annual leave can be carried over for up to 5 days into the next calendar year.
------------------------------------------------------------
Retrieved chunk 2
Source: documents/compensation_policy.txt
DataWise Co. Compensation Policy

Salary is paid on the last working day of each month.

Performance bonuses are reviewed once per year in December.

Employees may receive an annual salary adjustment based on company performance, individual performance, and market benchmarks.

Overtime pay is available only for non-managerial employees and must be approved by a manager before the overtime work

## 4. Generate LLM Response

In [12]:
# Import packages
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Initialise Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Create prompt template
prompt = ChatPromptTemplate.from_template("""
You are an HR policy assistant.

Answer the user's question using only the policy context below.

Rules:
- Do not use outside knowledge.
- If the answer is not in the context, say:
  "I could not find this information in the available company policies."
- Keep the answer concise.
- Mention the source policy file when possible.

Policy context:
{context}

User question:
{question}
""")

In [13]:
# Ask a question
question = "Do I need a medical certificate for sick leave?"

# Retrieve relevant document chunks
relevant_docs = retriever.invoke(question)

# Combine retrieved chunks into one context string
context = "\n\n".join(
    [
        f"Source: {doc.metadata['source']}\n"
        f"{doc.page_content}"
        for doc in relevant_docs
    ]
)

# Inspect retrieved context before sending it to Gemini
print("Retrieved context:")
print(context)

Retrieved context:
Source: documents/leave_policy.txt
DataWise Co. Leave Policy

Full-time employees receive 10 days of annual leave per year after completing probation.

Employees receive 15 days of paid sick leave per year.

Sick leave of 3 consecutive days or more requires a medical certificate.

Employees should submit annual leave requests at least 7 days in advance through the HR system.

Unused annual leave can be carried over for up to 5 days into the next calendar year.

Source: documents/compensation_policy.txt
DataWise Co. Compensation Policy

Salary is paid on the last working day of each month.

Performance bonuses are reviewed once per year in December.

Employees may receive an annual salary adjustment based on company performance, individual performance, and market benchmarks.

Overtime pay is available only for non-managerial employees and must be approved by a manager before the overtime work begins.


In [14]:
# Add context and question to prompt template
messages = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

# Send prompt to Gemini
response = llm.invoke(messages)

# Print Gemini's answer
print(response.content)

Yes, sick leave of 3 consecutive days or more requires a medical certificate. (Source: documents/leave_policy.txt)


## 5. Convert to Function

In [ ]:
# Convert to function
def ask_policy_question(question: str) -> str:
    """
    Retrieve relevant policy chunks, send them to Gemini,
    and return Gemini's answer.
    """

    # Retrieve relevant chunks
    relevant_docs = retriever.invoke(question)

    # Combine retrieved chunks into context
    context = "\n\n".join(
        [
            f"Source: {doc.metadata['source']}\n"
            f"{doc.page_content}"
            for doc in relevant_docs
        ]
    )

    # Add context and question to prompt template
    messages = prompt.invoke(
        {
            "context": context,
            "question": question
        }
    )

    # Send completed prompt to Gemini
    response = llm.invoke(messages)

    # Return answer text
    return response.content

In [18]:
# Test function
question = "Who is eligible for health insurance?"

answer = ask_policy_question(question)

print(answer)

Full-time employees receive health insurance after completing probation. (Source: documents/benefits_policy.txt)
